In [1]:
#The code in this block comes directly from data_sort_and_split.ipynb
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

heart_data = pd.read_csv("heart.csv")

heart_data['Sex_F'] = heart_data['Sex'].map({'M': 0, 'F': 1})
heart_data['ExerciseAngina'] = heart_data['ExerciseAngina'].map({'N': 0, 'Y': 1})
heart_data = heart_data.drop(columns=['Sex'])

heart_data['ChestPainType'] = pd.Categorical(heart_data['ChestPainType'], categories=['ASY', 'ATA', 'NAP', 'TA'])
heart_data['RestingECG'] = pd.Categorical(heart_data['RestingECG'], categories=['Normal', 'LVH', 'ST'])
heart_data['ST_Slope'] = pd.Categorical(heart_data['ST_Slope'], categories=['Up', 'Flat', 'Down'])

categorical_cols = ['ChestPainType', 'RestingECG', 'ST_Slope']
heart_data = pd.get_dummies(heart_data, columns=categorical_cols, drop_first=True, dtype=int)

feature_matrix = heart_data.drop("HeartDisease", axis=1)
target_labels = heart_data["HeartDisease"]

features_train, features_test, targets_train, targets_test = train_test_split(
    feature_matrix,
    target_labels,
    test_size=0.20,
    random_state=42,
    stratify= target_labels
)
#scaling performed because KNN is distance based so without scaling larger values can disturb the distance calculation
scaler = StandardScaler()
scaler.fit(features_train)
#transforming both sets using the training parameters
features_train_scaled = scaler.transform(features_train)
features_test_scaled = scaler.transform(features_test)


In [2]:
#K-Nearest Neighbors model creation
#using 1-20 values of k to know which performs the best
k_values = range(1,21)
accuracies = []
for k in k_values:
    knn_model = KNeighborsClassifier(n_neighbors=k)
    knn_model.fit(features_train_scaled,targets_train)
    predictions = knn_model.predict(features_test_scaled)
    accuracy = accuracy_score(targets_test,predictions)
    accuracies.append(accuracy)
best_k = k_values[accuracies.index(max(accuracies))]
print("Best k:",best_k)
print("Best Validation accuracy :",max(accuracies))

Best k: 7
Best Validation accuracy : 0.9130434782608695


In [3]:
#Model Training for KNN
#Rebuild the model using the best k values which is found in the previous block of code
final_knn_model = KNeighborsClassifier(n_neighbors=best_k)
final_knn_model.fit(features_train_scaled,targets_train)
final_prediction = final_knn_model.predict(features_test_scaled)

print("Final accuracy: ", accuracy_score(targets_test,final_prediction))
print("Confusion matrix: " )
print(confusion_matrix(targets_test,final_prediction))
print("Classification report: ")
print(classification_report(targets_test,final_prediction))

Final accuracy:  0.9130434782608695
Confusion matrix: 
[[73  9]
 [ 7 95]]
Classification report: 
              precision    recall  f1-score   support

           0       0.91      0.89      0.90        82
           1       0.91      0.93      0.92       102

    accuracy                           0.91       184
   macro avg       0.91      0.91      0.91       184
weighted avg       0.91      0.91      0.91       184

